# TECLA · demostración

Extrae los candidatos a término de un corpus, describe el corpus y evalúa su representatividad, sin instalar nada. Pulsa ▶ en las dos celdas, por orden.

> La demostración incluye unos textos sintéticos de prueba sobre lingüística de corpus, generados con inteligencia artificial generativa, para que puedas probarla sin cargar nada. También puedes cargar textos propios: se copian a esta sesión de Google Colab, así que no subas documentos confidenciales ni con derechos de autor. Para trabajar con tu propio corpus, [recomendamos instalar TECLA](https://github.com/isabel-mm/TECLA).

In [ ]:
#@title 1 · Instalar TECLA (unos dos minutos) { display-mode: "form" }
import subprocess, sys, os
from IPython.display import display, HTML

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"No se pudo ejecutar: {cmd}")

REPO = "/content/TECLA"
if os.path.isdir(REPO):
    _run(f"git -C {REPO} pull -q")      # ya descargada: se actualiza
else:
    _run(f"git clone -q https://github.com/isabel-mm/TECLA.git {REPO}")
_run(f"{sys.executable} -m pip install -q -e {REPO}")
_run(f"{sys.executable} -m spacy download es_core_news_md")
_run(f"{sys.executable} -m spacy download en_core_web_lg")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
display(HTML('<div style="font:600 15px sans-serif;color:#0d9488;padding:8px 0">✓ TECLA está lista. Pulsa ▶ en la celda siguiente.</div>'))

In [ ]:
#@title 2 · Abrir TECLA { display-mode: "form" }
import csv, shutil, sys
from collections import Counter
from pathlib import Path

import ipywidgets as w
import pandas as pd
from IPython.display import HTML, display

REPO = globals().get("REPO", "/content/TECLA")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
TRABAJO = Path(globals().get("TRABAJO", "/content/mi_corpus"))

from pipeline.pipeline import run_tecla_pipeline
from tecla.extractor.tbx_export import export_tbx
from tecla.corpus.cleaning import get_language_stopwords
from tecla.evaluation.corpus_metrics import (
    compute_bootstrap_stability, compute_dp, compute_hapax_ratio, compute_heaps,
    compute_representativity_score, dp_stats)

VIOLETA, TINTA, GRIS, LAVANDA, BORDE = "#7c3aed", "#1e1b4b", "#64748b", "#f5f3ff", "#ddd6fe"
PERFILES = {"Frecuencia": {"tfidf": 0.6, "ncvalue": 0.4},
            "Equilibrado": {"tfidf": 0.5, "ncvalue": 0.5},
            "Complejidad": {"tfidf": 0.2, "ncvalue": 0.8}}
LENGUAS = {"auto": None, "español": "es", "inglés": "en"}
EJEMPLOS = {"Ejemplo en inglés": "en", "Ejemplo en español": "es"}


def extraer(origen, subidos, lengua, monolexicos, perfil):
    """Ejecuta el flujo de TECLA y devuelve (cifras, candidatos, documentos, csv, tbx, evaluación)."""
    corpus = TRABAJO / "corpus"
    shutil.rmtree(TRABAJO, ignore_errors=True)
    corpus.mkdir(parents=True)
    if origen in EJEMPLOS:
        for f in sorted((Path(REPO) / "demo" / "ejemplos" / EJEMPLOS[origen]).glob("*.txt")):
            shutil.copy(f, corpus / f.name)
        lengua = "inglés" if EJEMPLOS[origen] == "en" else "español"
    else:
        if not subidos:
            raise ValueError("Sube al menos un texto (PDF o TXT) con el botón «Subir textos».")
        for nombre, contenido in subidos.items():
            (corpus / nombre).write_bytes(contenido)

    docs, detectada = [], []
    res = run_tecla_pipeline(
        corpus_path=corpus, language=LENGUAS[lengua],
        cleaning_options={"structure": True, "remove_bibliography_flag": True,
                          "remove_captions_flag": True, "remove_section_headers_flag": True},
        extraction_options={"multiword_only": not monolexicos, "max_term_length": 5,
                            "weights": PERFILES[perfil]},
        on_progress=lambda m: None,
        on_corpus_ready=lambda d, carpeta, lang: (docs.extend(d), detectada.append(lang)),
    )
    if not res:
        raise ValueError("No se ha extraído ningún candidato. Comprueba la lengua y los textos.")
    res.sort(key=lambda t: -t["terminologicity"])

    candidatos = pd.DataFrame([{
        "Candidato": t["lemma"], "Variantes": " · ".join(t.get("variants", [])) or "—",
        "Terminologicidad": round(t["terminologicity"], 4), "TF-IDF": round(t.get("tf_idf", 0), 2),
        "C-value": round(t.get("c_value", 0), 2), "NC-value": round(t.get("nc_value", 0), 2),
        "Frecuencia": t["frequency"], "Documentos": t["doc_frequency"]} for t in res])
    candidatos.index += 1

    try:
        vacias = get_language_stopwords(detectada[0] if detectada else "en")
    except Exception:
        vacias = set()
    filas, lemas, total, oraciones = [], set(), 0, 0
    for d in docs:
        cuenta = Counter(t["lemma"] for t in d["tokens"])
        n, n_or = len(d["tokens"]), len(d.get("sentences", []))
        lemas |= set(cuenta); total += n; oraciones += n_or
        palabras = [t["text"].lower() for t in d["tokens"] if t["text"].isalpha()]
        filas.append({"Documento": d["doc_id"], "Tokens": n, "Oraciones": n_or, "Tipos": len(cuenta),
                      "TTR": round(len(cuenta) / n, 4) if n else 0,
                      "Hápax": round(sum(1 for c in cuenta.values() if c == 1) / len(cuenta), 3) if cuenta else 0,
                      "Dens. léx.": round(sum(1 for p in palabras if p not in vacias) / len(palabras), 3) if palabras else 0,
                      "Long. or.": round(n / n_or, 1) if n_or else 0})
    filas.append({"Documento": "Corpus completo", "Tokens": total, "Oraciones": oraciones, "Tipos": len(lemas),
                  "TTR": round(len(lemas) / total, 4) if total else 0, "Hápax": float("nan"),
                  "Dens. léx.": float("nan"), "Long. or.": round(total / oraciones, 1) if oraciones else 0})
    cifras = {"documentos": len(docs), "tokens": total, "tipos": len(lemas), "candidatos": len(res)}

    ruta_csv, ruta_tbx = TRABAJO / "candidatos_tecla.csv", TRABAJO / "candidatos_tecla.tbx"
    campos = ["term", "lemma", "variants", "patterns", "terminologicity", "tf_idf",
              "c_value", "nc_value", "frequency", "freq_rel", "doc_frequency"]
    with open(ruta_csv, "w", encoding="utf-8", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=campos); wr.writeheader()
        for t in res:
            fila = {k: t.get(k, "") for k in campos}
            fila["patterns"] = ";".join(t.get("patterns", [])); fila["variants"] = ";".join(t.get("variants", []))
            wr.writerow(fila)
    export_tbx(res, ruta_tbx, language=(detectada[0] if detectada else "en"))
    return cifras, candidatos, pd.DataFrame(filas), ruta_csv, ruta_tbx, evaluar(res, docs)


def evaluar(res, docs):
    """Representatividad terminológica del corpus: los mismos cálculos que la pantalla Evaluación."""
    doc_stats = [{"doc_id": d["doc_id"], "tokens": len(d["tokens"]),
                  "sentences": len(d.get("sentences", [])),
                  "types": len({t["lemma"] for t in d["tokens"]})} for d in docs]
    heaps = compute_heaps(docs)
    hapax = compute_hapax_ratio(docs)
    dp = compute_dp(res, doc_stats)
    resumen_dp = dp_stats(dp, res)
    estabilidad = compute_bootstrap_stability(res, doc_stats)
    irt = compute_representativity_score(heaps, hapax, resumen_dp, stability=estabilidad)
    tabla_dp = pd.DataFrame([{"Candidato": t["lemma"], "Frecuencia": t["frequency"],
                              "Documentos": t["doc_frequency"], "DP": round(dp.get(t["lemma"], float("nan")), 3)}
                             for t in res[:50]])
    tabla_dp.index += 1
    return {"irt": irt, "heaps": heaps, "hapax": hapax, "dp": resumen_dp,
            "estabilidad": estabilidad, "tabla_dp": tabla_dp,
            "tokens": sum(d["tokens"] for d in doc_stats), "documentos": len(doc_stats)}


def _panel_evaluacion(ev):
    irt = ev["irt"]
    barras = "".join(
        f'<div style="display:flex;align-items:center;gap:10px;margin:5px 0;font:13px sans-serif;color:{TINTA}">'
        f'<span style="width:110px">{nombre.capitalize()}</span>'
        f'<span style="flex:1;max-width:260px;height:9px;background:{LAVANDA};border-radius:5px;overflow:hidden">'
        f'<span style="display:block;height:9px;width:{valor:.0f}%;background:{VIOLETA}"></span></span>'
        f'<span style="width:40px;text-align:right">{valor:.0f}</span></div>'
        for nombre, valor in irt.get("components", {}).items())
    beta = ev["heaps"].get("beta")
    est = ev["estabilidad"].get("stability_index")
    datos = " · ".join(x for x in [
        f"β de Heaps: {beta:.3f}" if beta is not None else "",
        f"hápax: {ev['hapax']['ratio']:.1%} de los lemas",
        f"DP medio (50 primeros candidatos): {ev['dp'].get('mean', float('nan')):.3f}",
        f"estabilidad: {est:.3f}" if est is not None else ""] if x)
    aviso = ""
    if ev["tokens"] < 100_000 or ev["documentos"] < 10:
        aviso = ('<div style="background:#fff7ed;border:1px solid #fed7aa;border-radius:8px;padding:9px 12px;'
                 'font:13px sans-serif;color:#7c2d12;margin-bottom:12px">Con un corpus tan pequeño estas cifras '
                 'son solo ilustrativas: sirven para ver qué calcula TECLA, no para valorar el corpus.</div>')
    return (f'{aviso}<div style="display:flex;gap:28px;flex-wrap:wrap;align-items:flex-start">'
            f'<div><div style="font:600 11px sans-serif;letter-spacing:.08em;color:{GRIS};text-transform:uppercase">'
            f'Índice de representatividad terminológica</div>'
            f'<div style="font:800 46px sans-serif;color:{irt["color"]};line-height:1.1">{irt["score"]}'
            f'<span style="font:400 18px sans-serif;color:{GRIS}"> / 100</span></div>'
            f'<div style="font:600 14px sans-serif;color:{irt["color"]}">El corpus es {irt["label"]}</div></div>'
            f'<div style="flex:1;min-width:300px">{barras}</div></div>'
            f'<div style="font:13px sans-serif;color:{GRIS};margin:10px 0 4px">{datos}</div>')


def _curva_heaps(heaps):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6.2, 3.1))
    ax.plot(heaps["tokens"], heaps["types"], color=VIOLETA, linewidth=2, label="Tipos observados")
    if heaps.get("predicted"):
        ax.plot(heaps["tokens"], heaps["predicted"], color="#0d9488", linewidth=1.2, linestyle="--",
                label=f"Ley de Heaps (β = {heaps['beta']:.3f})")
    ax.set_xlabel("Tokens"); ax.set_ylabel("Tipos (lemas distintos)")
    ax.set_title("Crecimiento del vocabulario", fontsize=11, loc="left")
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()


def _miles(n):
    return f"{int(n):,}".replace(",", " ")

def _tarjetas(cifras):
    cajas = "".join(
        f'<div style="flex:1;min-width:130px;background:#fff;border:1px solid #e2e8f0;border-radius:10px;padding:12px 14px">'
        f'<div style="font:700 22px sans-serif;color:{TINTA}">{_miles(v)}</div>'
        f'<div style="font:13px sans-serif;color:{GRIS}">{k}</div></div>' for k, v in cifras.items())
    return f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:6px 0 14px">{cajas}</div>'

def _tabla(df, n=None):
    df = df if n is None else df.head(n)
    return (df.style.set_table_styles([
        {"selector": "th", "props": f"background:{LAVANDA};color:{GRIS};font:600 12px sans-serif;text-align:right;padding:8px 12px;border-bottom:1px solid {BORDE}"},
        {"selector": "td", "props": f"font:14px sans-serif;color:{TINTA};text-align:right;padding:7px 12px;border-bottom:1px solid #eef1f4"},
        {"selector": "td.col0, th.col0", "props": "text-align:left"},
    ]).format(precision=3, na_rep="—", subset=[c for c in ["DP", "Hápax", "Dens. léx."] if c in df.columns])
      .format(precision=1, subset=[c for c in ["Long. or."] if c in df.columns]).bar(subset=[c for c in ["Terminologicidad"] if c in df.columns], color="#ddd6fe", vmin=0, vmax=1)
      .format(precision=4, subset=[c for c in ["Terminologicidad", "TTR"] if c in df.columns])
      .format(precision=2, subset=[c for c in ["TF-IDF", "C-value", "NC-value"] if c in df.columns]))


# ── Interfaz ──
estilo = {"description_width": "0px"}
origen = w.ToggleButtons(options=list(EJEMPLOS) + ["Cargar textos propios"], style={"button_width": "178px"})
subida = w.FileUpload(accept=".txt,.pdf", multiple=True, description="Subir textos",
                      layout=w.Layout(display="none", width="170px"))
lengua = w.ToggleButtons(options=list(LENGUAS), style={"button_width": "92px"})
perfil = w.ToggleButtons(options=list(PERFILES), value="Equilibrado", style={"button_width": "110px"})
mono = w.Checkbox(value=True, description="Incluir candidatos monoléxicos", indent=False)
boton = w.Button(description="▶  Ejecutar", button_style="primary",
                 layout=w.Layout(width="170px", height="40px"))
boton.style.button_color = VIOLETA
estado = w.HTML()
salida = w.Output()
b_csv = w.Button(description="⬇  Descargar CSV", layout=w.Layout(width="170px", display="none"))
b_tbx = w.Button(description="⬇  Descargar TBX", layout=w.Layout(width="170px", display="none"))
_rutas = {}

def _rotulo(t):
    return w.HTML(f'<div style="font:600 11px sans-serif;letter-spacing:.08em;color:{GRIS};text-transform:uppercase;margin-top:8px">{t}</div>')

def _al_cambiar_origen(cambio=None):
    # Con los ejemplos la lengua va fijada por el ejemplo; con textos propios se elige.
    propio = origen.value == "Cargar textos propios"
    subida.layout.display = "inline-flex" if propio else "none"
    lengua.disabled = False
    lengua.value = "auto" if propio else ("inglés" if EJEMPLOS[origen.value] == "en" else "español")
    lengua.disabled = not propio
    nota_lengua.value = "" if propio else (
        f'<span style="font:12px sans-serif;color:{GRIS}">La lengua la fija el ejemplo. '
        f'Elige «Cargar textos propios» para escogerla tú.</span>')
nota_lengua = w.HTML()
origen.observe(_al_cambiar_origen, names="value")
_al_cambiar_origen()

def _subidos():
    v = subida.value
    if isinstance(v, dict):                      # ipywidgets 7
        return {k: x["content"] for k, x in v.items()}
    return {x["name"]: bytes(x["content"]) for x in v}   # ipywidgets 8

def _al_ejecutar(_):
    boton.disabled = True
    b_csv.layout.display = b_tbx.layout.display = "none"
    estado.value = f'<span style="font:14px sans-serif;color:{GRIS}">Procesando los textos…</span>'
    salida.clear_output()
    try:
        cifras, cand, docs, ruta_csv, ruta_tbx, ev = extraer(origen.value, _subidos(), lengua.value, mono.value, perfil.value)
        _rutas.update(csv=ruta_csv, tbx=ruta_tbx)
        with salida:
            display(HTML(_tarjetas(cifras)))
            pestañas = w.Tab()
            t1, t2, t3 = w.Output(), w.Output(), w.Output()
            with t1:
                if len(cand) > 100:
                    display(HTML(f'<div style="font:13px sans-serif;color:{GRIS}">Se muestran los 100 primeros; el CSV los incluye todos.</div>'))
                display(_tabla(cand, 100))
            with t2:
                display(HTML(f'<div style="font:13px sans-serif;color:{GRIS};margin-bottom:6px">Tipos: lemas distintos · '
                             f'TTR: tipos entre tokens · Hápax: proporción de tipos que aparecen una sola vez · '
                             f'Dens. léx.: palabras de contenido entre las palabras · Long. or.: tokens por oración</div>'))
                display(_tabla(docs))
            with t3:
                display(HTML(_panel_evaluacion(ev)))
                _curva_heaps(ev["heaps"])
                display(HTML(f'<div style="font:600 13px sans-serif;color:{TINTA};margin:10px 0 4px">'
                             f'Dispersión de los 50 primeros candidatos (DP: 0 = repartido por todos los '
                             f'documentos; 1 = concentrado en uno)</div>'))
                display(_tabla(ev["tabla_dp"]))
            pestañas.children = [t1, t2, t3]
            pestañas.set_title(0, "Candidatos a término"); pestañas.set_title(1, "Corpus")
            pestañas.set_title(2, "Evaluación")
            display(pestañas)
        estado.value = ""
        b_csv.layout.display = b_tbx.layout.display = "inline-flex"
    except Exception as e:
        estado.value = f'<span style="font:14px sans-serif;color:#dc2626">{e}</span>'
    finally:
        boton.disabled = False
boton.on_click(_al_ejecutar)

def _descargar(clave):
    def f(_):
        from google.colab import files
        files.download(str(_rutas[clave]))
    return f
b_csv.on_click(_descargar("csv")); b_tbx.on_click(_descargar("tbx"))

cabecera = w.HTML(
    f'<div style="display:flex;align-items:center;gap:12px;margin-bottom:4px">'
    f'<div style="width:40px;height:40px;border-radius:10px;background:{VIOLETA};color:#fff;display:grid;place-items:center;font:800 20px sans-serif">T</div>'
    f'<div><div style="font:800 20px sans-serif;color:{TINTA};letter-spacing:.04em">TECLA</div>'
    f'<div style="font:12px sans-serif;color:{GRIS}">Term Extraction and Corpus Linguistics Application</div></div></div>')
panel = w.VBox([cabecera, _rotulo("Corpus"), w.HBox([origen, subida]), _rotulo("Lengua"), lengua, nota_lengua,
                _rotulo("Perfil de puntuación"), perfil, mono,
                w.HBox([boton, b_csv, b_tbx], layout=w.Layout(margin="10px 0 0 0")), estado, salida],
               layout=w.Layout(padding="18px", border=f"1px solid {BORDE}"))
display(panel)

**Corpus**: los ejemplos son seis textos sintéticos de prueba sobre lingüística de corpus, generados con inteligencia artificial generativa (Claude). **Perfil**: peso del *TF-IDF* frente al *NC-value* (0,6 / 0,5 / 0,2). **Evaluación**: saturación (ley de Heaps), proporción de hápax, dispersión (DP) y estabilidad por remuestreo, resumidas en un índice de 0 a 100. La revisión de los candidatos, la cobertura frente a una lista de referencia y el contraste con un corpus de referencia están en la aplicación de escritorio. Licencia MIT · [github.com/isabel-mm/TECLA](https://github.com/isabel-mm/TECLA)